# Tema 05: Dash

**Taller:** Análisis y Visualización Interactiva en Python
**Duración estimada de esta sesión:** 60 minutos
**Herramienta principal:** Dash (Plotly)
**Modalidad de práctica:** Google Colab

---

> 📌 **Nota para el profesor:** esta notebook está diseñada para proyectarse y ejecutarse en vivo. Las secciones marcadas como **Práctica guiada** se resuelven junto con el grupo; las marcadas como **Práctica independiente** las resuelven los participantes en sus propias copias de la notebook (`Archivo → Guardar una copia en Drive`).

## 🎯 Objetivos de aprendizaje

Al finalizar este tema, el participante será capaz de:

- Explicar el modelo de programación reactiva de Dash: `layout` (qué se ve) + `callbacks` (cómo reacciona).
- Construir el layout de una app con componentes de `dash.html` (estructura) y `dash.dcc` (controles interactivos y gráficos).
- Conectar controles (`dcc.Dropdown`, `dcc.Slider`) a gráficos de Plotly mediante `@app.callback` con `Input`/`Output`.
- Ejecutar y visualizar una app Dash directamente dentro de Google Colab.
- Construir un mini-dashboard con dos gráficos que reaccionan a los mismos filtros.

## 🧠 Contenido teórico

### 1. De una figura a una aplicación

Con Plotly (Tema 04) generamos **una** figura interactiva a la vez. **Dash** es un framework (también de Plotly) para construir **aplicaciones web completas** de análisis de datos usando solo Python — sin escribir HTML, CSS ni JavaScript. Es el equivalente en Python a herramientas como Shiny (R) o Streamlit.

Un caso típico: quieres que la persona que usa tu análisis pueda **elegir** un país, un rango de fechas o una categoría con un menú desplegable, y que el gráfico se actualice automáticamente — sin volver a ejecutar código.

### 2. Los dos bloques de toda app Dash

1. **`app.layout`**: define **qué se ve** en la página — una jerarquía de componentes, igual que HTML pero escrito en Python:
   - `dash.html` (`html.H1`, `html.Div`, `html.P`, ...): estructura visual, equivalente a etiquetas HTML.
   - `dash.dcc` ("Dash Core Components"): controles interactivos (`dcc.Dropdown`, `dcc.Slider`, `dcc.RangeSlider`, `dcc.DatePickerRange`) y el componente `dcc.Graph` para mostrar figuras de Plotly.

2. **`@app.callback`**: define **cómo reacciona** la app — una función de Python que se ejecuta automáticamente cada vez que cambia un `Input`, y actualiza uno o más `Output`.

```python
@app.callback(
    Output("mi-grafico", "figure"),   # QUÉ se actualiza (componente "mi-grafico", propiedad "figure")
    Input("mi-dropdown", "value"),    # QUÉ dispara la actualización (componente "mi-dropdown", propiedad "value")
)
def actualizar_grafico(valor_seleccionado):
    # ... construir y devolver una figura de Plotly según valor_seleccionado
    return figura
```

Cada componente tiene un `id` único — así es como el callback sabe con qué componente conectarse. Este patrón (`Input` dispara → función procesa → `Output` actualiza) es la esencia de la **programación reactiva**.

### 3. Ejecutar Dash en Google Colab

Desde la versión 2.11, Dash detecta automáticamente si se ejecuta dentro de un notebook (Jupyter/Colab) y permite mostrar la app **dentro** de la notebook con:

```python
app.run(jupyter_mode="inline")   # la app aparece incrustada debajo de la celda
# también: jupyter_mode="external" (abre en una pestaña/URL aparte)
```

**Importante para la sesión en vivo:** solo se puede tener un servidor activo por puerto a la vez. Antes de ejecutar el siguiente ejemplo, detén la celda anterior (botón ■) o usa un puerto distinto (`port=8051`, `8052`, ...), como haremos en esta notebook.

## ⚙️ Configuración del entorno

Cargaremos `tema05_energia_renovable.csv`: capacidad instalada y producción (sintéticas) de energías renovables por país, fuente de energía y año (2015-2024).

In [ ]:
# === Carga del dataset ===
# Opción 1 (recomendada una vez publicado el repositorio del taller):
# reemplaza <usuario>/<repositorio> por la ruta real de tu repo de GitHub
# y ejecuta esta celda. Usa el botón "Raw" de GitHub para obtener la URL.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<usuario>/<repositorio>/main/"
    "datasets/tema05_energia_renovable.csv"
)

import pandas as pd

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print("Datos cargados desde GitHub ✅  ->", df.shape)
except Exception as e:
    print("No se pudo leer desde GitHub todavía (repo no configurado o sin internet).")
    print("Sube manualmente el archivo 'tema05_energia_renovable.csv' cuando se te solicite.")
    try:
        from google.colab import files
        subido = files.upload()  # selecciona tema05_energia_renovable.csv
        df = pd.read_csv(list(subido.keys())[0])
    except ImportError:
        # Fuera de Colab (por ejemplo, ejecución local de prueba):
        df = pd.read_csv("tema05_energia_renovable.csv")

df.head()

## 🧭 Práctica guiada

Instala Dash si aún no está disponible en el entorno (en Colab normalmente ya viene preinstalado; si no, descomenta la siguiente línea).

In [ ]:
# !pip install dash -q

from dash import Dash, dcc, html, Input, Output
import plotly.express as px

### Paso 1 · La app más simple posible (layout estático)

In [ ]:
app1 = Dash(__name__)

app1.layout = html.Div([
    html.H1("Mi primera app Dash"),
    html.P("Este texto viene de un componente html.P — sin necesidad de escribir HTML."),
])

# En Colab, ejecuta esta celda y la app aparecerá incrustada debajo.
app1.run(jupyter_mode="inline", port=8050)

### Paso 2 · Agregar un gráfico (`dcc.Graph`) con datos fijos

In [ ]:
produccion_2024 = (
    df[df["anio"] == 2024]
    .groupby("fuente_energia", as_index=False)["produccion_gwh"].sum()
)
figura_inicial = px.bar(produccion_2024, x="fuente_energia", y="produccion_gwh",
                         title="Producción de energía renovable 2024 (todos los países)")

app2 = Dash(__name__)
app2.layout = html.Div([
    html.H1("Producción de energía renovable"),
    dcc.Graph(id="grafico-produccion", figure=figura_inicial),
])

app2.run(jupyter_mode="inline", port=8051)

### Paso 3 · Interactividad — un `Dropdown` que controla el gráfico

In [ ]:
paises_disponibles = sorted(df["pais"].unique())

app3 = Dash(__name__)

app3.layout = html.Div([
    html.H1("Producción de energía renovable por país"),
    html.Label("Selecciona un país:"),
    dcc.Dropdown(
        id="selector-pais",
        options=[{"label": p, "value": p} for p in paises_disponibles],
        value="México",
    ),
    dcc.Graph(id="grafico-produccion-pais"),
])

@app3.callback(
    Output("grafico-produccion-pais", "figure"),
    Input("selector-pais", "value"),
)
def actualizar_grafico_pais(pais_seleccionado):
    datos = df[df["pais"] == pais_seleccionado]
    resumen = datos.groupby(["anio", "fuente_energia"], as_index=False)["produccion_gwh"].sum()
    fig = px.line(resumen, x="anio", y="produccion_gwh", color="fuente_energia",
                  title=f"Producción de energía renovable — {pais_seleccionado} (2015-2024)")
    return fig

app3.run(jupyter_mode="inline", port=8052)

### Paso 4 · Dos controles y dos gráficos (mini-dashboard)

In [ ]:
fuentes_disponibles = sorted(df["fuente_energia"].unique())

app4 = Dash(__name__)

app4.layout = html.Div([
    html.H1("Panel de energía renovable"),
    html.Div([
        html.Div([
            html.Label("País:"),
            dcc.Dropdown(
                id="dd-pais",
                options=[{"label": p, "value": p} for p in paises_disponibles],
                value="México",
            ),
        ], style={"width": "45%", "display": "inline-block"}),

        html.Div([
            html.Label("Rango de años:"),
            dcc.RangeSlider(
                id="slider-anios", min=2015, max=2024, step=1, value=[2018, 2024],
                marks={a: str(a) for a in range(2015, 2025, 3)},
            ),
        ], style={"width": "50%", "display": "inline-block", "marginLeft": "4%"}),
    ]),

    dcc.Graph(id="grafico-linea"),
    dcc.Graph(id="grafico-pastel"),
])

@app4.callback(
    Output("grafico-linea", "figure"),
    Output("grafico-pastel", "figure"),
    Input("dd-pais", "value"),
    Input("slider-anios", "value"),
)
def actualizar_dashboard(pais, rango_anios):
    datos = df[(df["pais"] == pais) & (df["anio"].between(rango_anios[0], rango_anios[1]))]

    resumen_anual = datos.groupby(["anio", "fuente_energia"], as_index=False)["produccion_gwh"].sum()
    fig_linea = px.line(resumen_anual, x="anio", y="produccion_gwh", color="fuente_energia",
                         title=f"Producción por fuente — {pais}")

    resumen_fuente = datos.groupby("fuente_energia", as_index=False)["produccion_gwh"].sum()
    fig_pastel = px.pie(resumen_fuente, names="fuente_energia", values="produccion_gwh",
                         title=f"Participación por fuente — {pais} ({rango_anios[0]}-{rango_anios[1]})")

    return fig_linea, fig_pastel

app4.run(jupyter_mode="inline", port=8053)

## ✍️ Práctica independiente

Parte del código del **Paso 4** (`app4`). Usa un puerto nuevo (por ejemplo `8060`) para cada ejercicio.

**Ejercicio 1.** Agrega un segundo `dcc.Dropdown` (`id='dd-fuente'`, con opción `'Todas'` + cada fuente de `fuentes_disponibles`) que filtre adicionalmente por `fuente_energia` cuando no sea `'Todas'`.

In [ ]:
# TODO: tu código aquí (crea una nueva app, p. ej. app_ej1, en el puerto 8060)

**Ejercicio 2.** Agrega una tercera salida: una tarjeta de texto (`html.Div` o `html.H3`) que muestre la **capacidad instalada total** (`capacidad_instalada_mw`) para el país y rango de años seleccionados. Pista: agrega un `Output` adicional y un componente con `id='texto-capacidad'`.

In [ ]:
# TODO: tu código aquí (usa el puerto 8061)

**Ejercicio 3.** Cambia el gráfico de línea para que sea un `px.area` (área apilada) en lugar de `px.line`.

In [ ]:
# TODO: tu código aquí (usa el puerto 8062)

**Ejercicio 4 (reto).** Agrega un `dcc.Dropdown` con `multi=True` para seleccionar **varios países a la vez**, y actualiza el gráfico de línea para comparar la producción total (suma de fuentes) entre los países seleccionados.

In [ ]:
# TODO: tu código aquí (usa el puerto 8063)

---
### ✅ Soluciones (referencia para el profesor)

In [ ]:
# Ejercicio 1
app_ej1 = Dash(__name__)
app_ej1.layout = html.Div([
    html.H1("Producción por país y fuente"),
    dcc.Dropdown(id="dd-pais", options=[{"label": p, "value": p} for p in paises_disponibles], value="México"),
    dcc.Dropdown(id="dd-fuente",
                 options=[{"label": "Todas", "value": "Todas"}] + [{"label": f, "value": f} for f in fuentes_disponibles],
                 value="Todas"),
    dcc.Graph(id="grafico-ej1"),
])

@app_ej1.callback(Output("grafico-ej1", "figure"), Input("dd-pais", "value"), Input("dd-fuente", "value"))
def actualizar_ej1(pais, fuente):
    datos = df[df["pais"] == pais]
    if fuente != "Todas":
        datos = datos[datos["fuente_energia"] == fuente]
    resumen = datos.groupby(["anio", "fuente_energia"], as_index=False)["produccion_gwh"].sum()
    return px.line(resumen, x="anio", y="produccion_gwh", color="fuente_energia",
                    title=f"{pais} — {fuente}")

app_ej1.run(jupyter_mode="inline", port=8060)

In [ ]:
# Ejercicio 2
app_ej2 = Dash(__name__)
app_ej2.layout = html.Div([
    html.H1("Producción y capacidad instalada"),
    dcc.Dropdown(id="dd-pais2", options=[{"label": p, "value": p} for p in paises_disponibles], value="México"),
    dcc.RangeSlider(id="slider-anios2", min=2015, max=2024, step=1, value=[2018, 2024],
                     marks={a: str(a) for a in range(2015, 2025, 3)}),
    html.H3(id="texto-capacidad"),
    dcc.Graph(id="grafico-ej2"),
])

@app_ej2.callback(
    Output("grafico-ej2", "figure"),
    Output("texto-capacidad", "children"),
    Input("dd-pais2", "value"),
    Input("slider-anios2", "value"),
)
def actualizar_ej2(pais, rango):
    datos = df[(df["pais"] == pais) & (df["anio"].between(rango[0], rango[1]))]
    resumen = datos.groupby(["anio", "fuente_energia"], as_index=False)["produccion_gwh"].sum()
    fig = px.line(resumen, x="anio", y="produccion_gwh", color="fuente_energia", title=pais)
    capacidad_total = datos[datos["anio"] == rango[1]]["capacidad_instalada_mw"].sum()
    texto = f"Capacidad instalada total en {rango[1]}: {capacidad_total:,.0f} MW"
    return fig, texto

app_ej2.run(jupyter_mode="inline", port=8061)

In [ ]:
# Ejercicio 3
app_ej3 = Dash(__name__)
app_ej3.layout = html.Div([
    dcc.Dropdown(id="dd-pais3", options=[{"label": p, "value": p} for p in paises_disponibles], value="México"),
    dcc.Graph(id="grafico-ej3"),
])

@app_ej3.callback(Output("grafico-ej3", "figure"), Input("dd-pais3", "value"))
def actualizar_ej3(pais):
    resumen = df[df["pais"] == pais].groupby(["anio", "fuente_energia"], as_index=False)["produccion_gwh"].sum()
    return px.area(resumen, x="anio", y="produccion_gwh", color="fuente_energia", title=f"{pais} (área apilada)")

app_ej3.run(jupyter_mode="inline", port=8062)

In [ ]:
# Ejercicio 4 (reto)
app_ej4 = Dash(__name__)
app_ej4.layout = html.Div([
    dcc.Dropdown(id="dd-paises4", options=[{"label": p, "value": p} for p in paises_disponibles],
                 value=["México", "Chile"], multi=True),
    dcc.Graph(id="grafico-ej4"),
])

@app_ej4.callback(Output("grafico-ej4", "figure"), Input("dd-paises4", "value"))
def actualizar_ej4(paises_sel):
    datos = df[df["pais"].isin(paises_sel)]
    resumen = datos.groupby(["anio", "pais"], as_index=False)["produccion_gwh"].sum()
    return px.line(resumen, x="anio", y="produccion_gwh", color="pais",
                    title="Comparación de producción total por país")

app_ej4.run(jupyter_mode="inline", port=8063)

## 🔎 Cierre y puente al siguiente tema

Con Dash construimos aplicaciones web completas, pero eso implica manejar un servidor, callbacks y más código. A veces solo necesitamos un gráfico interactivo **de alto rendimiento** (miles o millones de puntos) sin montar una aplicación completa — ahí es donde brilla **Bokeh** (Tema 06), especialmente para series de tiempo financieras o científicas.

## 📚 Recursos adicionales

- [Documentación oficial de Dash](https://dash.plotly.com/)
- [Tutorial oficial: Layout](https://dash.plotly.com/layout)
- [Tutorial oficial: Callbacks básicos](https://dash.plotly.com/basic-callbacks)
- [Dash en Jupyter/Colab (guía oficial)](https://dash.plotly.com/dash-in-jupyter)
- [Galería de apps Dash (Dash Gallery)](https://dash.gallery/Portal/)